# Inductive Node Classification on Reddit with SAGEConv

Node Classification on Reddit: GraphSAGE training on the large Reddit community interaction graph. This notebook implements the approach with `SAGEConv` inside a `K3SAGE` model, trained with the Adam optimizer for 50 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `SAGEConv` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import Reddit

title = "Inductive Node Classification with GraphSAGE"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Reddit(root="./data/Reddit")
data = dataset[0]
num_features = dataset.num_features
num_classes = dataset.num_classes

# 2. SAGEConv Model
class K3SAGE(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.SAGEConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.SAGEConv(hidden_channels, out_channels)
        self.dropout = layers.Dropout(0.5)

    def build(self, input_shape=None):
        self.conv1.build((None, self.conv1.in_channels))
        self.conv2.build((None, self.conv2.in_channels))
        self.built = True

    def call(self, inputs, edge_index=None, training=False):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = self.dropout(x, training=training)
        x = ops.relu(self.conv1(x, edge_index))
        x = self.dropout(x, training=training)
        return self.conv2(x, edge_index)

k3_model = K3SAGE(num_features, 128, num_classes)

# 3. Model Compilation
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

# 4. Generator & Training
def to_np(t, dtype=None):
    if hasattr(t, "cpu"):
        t = t.cpu()
    if hasattr(t, "detach"):
        t = t.detach()
    if hasattr(t, "numpy") and callable(t.numpy):
        t = t.numpy()
    return np.asarray(t, dtype=dtype)

def graph_data_generator():
    x = to_np(data.x, dtype=np.float32)
    edge_index = to_np(data.edge_index, dtype=np.int64)
    y = to_np(data.y, dtype=np.int64)
    mask = to_np(data.train_mask, dtype=np.float32)
    while True:
        yield (x, edge_index), y, mask

print(f"Training K3-Node GraphSAGE on {backend} backend...")
history = k3_model.fit(
    graph_data_generator(),
    steps_per_epoch=1,
    epochs=50,
    verbose=1,
)

# 5. Evaluation
out = k3_model((data.x, data.edge_index))
pred = ops.argmax(out, axis=-1)
test_mask = data.test_mask
test_acc = ops.mean(ops.cast(ops.cast(pred[test_mask], "int64") == ops.cast(data.y[test_mask], "int64"), "float32"))
print(f"Test Accuracy: {float(test_acc):.4f}")

print("\n✓ K3-Node GraphSAGE execution completed successfully!")